# Neo4j UC Federation: OAuth Bearer Token Connection

## Overview

This notebook validates Neo4j connectivity through Unity Catalog's JDBC path using **OAuth 2.0 Client Credentials** authentication instead of username/password. It targets Neo4j Enterprise Edition instances configured with OIDC authentication via Keycloak (or any OIDC provider).

**How it works:** The Neo4j Unity Catalog Connector JAR includes a Client Credentials authentication factory registered via Java SPI. When the JDBC URL contains `authn.supplier=cc`, the driver loads this factory instead of using basic auth. The factory acquires a bearer token from the OIDC token endpoint using the OAuth 2.0 Client Credentials grant, and presents the JWT to Neo4j. Token refresh is automatic.

**Key difference from notebook 01:** The `CREATE CONNECTION` statement passes the OAuth client ID and client secret through the standard `user`/`password` fields. The JDBC URL includes the auth supplier name and token endpoint. From Databricks' perspective, this looks like a normal JDBC connection with credentials.

**Prerequisites:**
- Neo4j EE deployed with OIDC authentication (e.g., via the azure-ee-template)
- A Keycloak (or other OIDC provider) deployment with a client configured for client credentials grant
- Neo4j Unity Catalog Connector JAR (with OAuth support) uploaded to a UC Volume
- Run `setup-keycloak-env.sh` to generate `.env.oauth`, then `setup-oauth.sh` to configure Databricks secrets

---

## Configuration

**Prerequisites**: Run `setup-keycloak-env.sh` to generate `.env.oauth` from a Keycloak deployment, then `setup-oauth.sh` to create Databricks secrets.

```bash
# Generate .env.oauth from keycloak-infra/.deployment.json
./setup-keycloak-env.sh /path/to/.deployment.json neo4j-host.example.com

# Edit .env.oauth to set JDBC_JAR_PATH and UC_CONNECTION_NAME if needed
# Then push secrets to Databricks
./setup-oauth.sh
```

The setup scripts read credentials from `.env.oauth` and store them in the `oauth-neo4j-uc-creds` secret scope:
- `host` - Neo4j host (bolt endpoint)
- `oauth_token_endpoint` - OIDC token endpoint URL (e.g., `https://keycloak.example.com/realms/neo4j/protocol/openid-connect/token`)
- `oauth_client_id` - OAuth client ID (e.g., `neo4j-client`)
- `oauth_client_secret` - OAuth client secret
- `connection_name` - Unity Catalog connection name
- `jdbc_jar_path` - Path to Neo4j Unity Catalog Connector JAR in UC Volume
- `database` - Neo4j database (optional, defaults to `neo4j`)

In [ ]:
# =============================================================================
# CONFIGURATION - Loaded from Databricks Secrets
# =============================================================================
import urllib.parse

SCOPE_NAME = "oauth-neo4j-uc-creds"

# Neo4j Connection
NEO4J_HOST = dbutils.secrets.get(SCOPE_NAME, "host")

# OAuth Credentials (replaces username/password)
OAUTH_TOKEN_ENDPOINT = dbutils.secrets.get(SCOPE_NAME, "oauth_token_endpoint")
OAUTH_CLIENT_ID = dbutils.secrets.get(SCOPE_NAME, "oauth_client_id")
OAUTH_CLIENT_SECRET = dbutils.secrets.get(SCOPE_NAME, "oauth_client_secret")

# Database defaults to "neo4j" if not set
try:
    NEO4J_DATABASE = dbutils.secrets.get(SCOPE_NAME, "database")
except:
    NEO4J_DATABASE = "neo4j"

# Unity Catalog Resources
JDBC_JAR_PATH = dbutils.secrets.get(SCOPE_NAME, "jdbc_jar_path")
UC_CONNECTION_NAME = dbutils.secrets.get(SCOPE_NAME, "connection_name")

JAVA_DEPENDENCIES = f'["{JDBC_JAR_PATH}"]'

# Build JDBC URL with OAuth parameters
# The authn.supplier=cc tells the Neo4j JDBC driver to use the Client Credentials factory
# The authn.cc.tokenEndpoint passes the OIDC token endpoint URL (must be URL-encoded)
ENCODED_TOKEN_ENDPOINT = urllib.parse.quote(OAUTH_TOKEN_ENDPOINT, safe='')

# bolt:// for standalone Neo4j EE (no routing, no TLS)
NEO4J_BOLT_URI = f"bolt://{NEO4J_HOST}:7687"
NEO4J_JDBC_URL_OAUTH = (
    f"jdbc:neo4j://{NEO4J_HOST}:7687/{NEO4J_DATABASE}"
    f"?enableSQLTranslation=true"
    f"&authn.supplier=cc"
    f"&authn.cc.tokenEndpoint={ENCODED_TOKEN_ENDPOINT}"
)

print("Configuration loaded from Databricks Secrets:")
print(f"  Secret Scope: {SCOPE_NAME}")
print(f"  Neo4j Host: {NEO4J_HOST}")
print(f"  Bolt URI: {NEO4J_BOLT_URI}")
print(f"  Token Endpoint: {OAUTH_TOKEN_ENDPOINT}")
print(f"  OAuth Client ID: {OAUTH_CLIENT_ID}")
print(f"  Connection Name: {UC_CONNECTION_NAME}")
print(f"  JDBC JAR Path: {JDBC_JAR_PATH}")
print(f"  JDBC URL (OAuth): {NEO4J_JDBC_URL_OAUTH}")

---

## Section 1: Environment Information

Capture cluster and runtime details for reproducibility.

In [ ]:
print("=" * 60)
print("ENVIRONMENT INFORMATION")
print("=" * 60)

print(f"\nSpark Version: {spark.version}")

try:
    dbr_version = spark.conf.get("spark.databricks.clusterUsageTags.sparkVersion")
    print(f"Databricks Runtime: {dbr_version}")
except:
    print("Databricks Runtime: Unable to determine")

import sys
print(f"Python Version: {sys.version}")

print(f"\nJDBC JAR Path: {JDBC_JAR_PATH}")
try:
    files = dbutils.fs.ls(JDBC_JAR_PATH.rsplit('/', 1)[0])
    file_names = [f.name for f in files]
    jdbc_jar_found = JDBC_JAR_PATH.split('/')[-1] in file_names
    print(f"JDBC JAR File Exists: {jdbc_jar_found}")
except Exception as e:
    print(f"JAR File Check Error: {e}")

---

## Section 2: Network Connectivity Test

Verify that the Databricks cluster can reach both Neo4j (Bolt) and the OIDC token endpoint (HTTPS). Run this before driver-level tests to isolate network issues early.

**Expected Result**: PASS for both endpoints.

In [ ]:
import subprocess
import time
from urllib.parse import urlparse

print("=" * 60)
print("TEST: Network Connectivity")
print("=" * 60)

# Test Neo4j Bolt port
print(f"\n--- Neo4j Bolt: {NEO4J_HOST}:7687 ---")
try:
    start_time = time.time()
    result = subprocess.run(
        ['nc', '-zv', NEO4J_HOST, '7687'],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=10
    )
    elapsed = (time.time() - start_time) * 1000
    if result.returncode == 0:
        print(f"[PASS] Neo4j Bolt reachable ({elapsed:.1f}ms)")
    else:
        print(f"[FAIL] Cannot reach Neo4j Bolt port")
except Exception as e:
    print(f"[FAIL] Error: {e}")

# Test OIDC token endpoint
parsed = urlparse(OAUTH_TOKEN_ENDPOINT)
token_host = parsed.hostname
token_port = str(parsed.port or 443)
print(f"\n--- Token Endpoint: {token_host}:{token_port} ---")
try:
    start_time = time.time()
    result = subprocess.run(
        ['nc', '-zv', token_host, token_port],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=10
    )
    elapsed = (time.time() - start_time) * 1000
    if result.returncode == 0:
        print(f"[PASS] Token endpoint reachable ({elapsed:.1f}ms)")
    else:
        print(f"[FAIL] Cannot reach token endpoint")
except Exception as e:
    print(f"[FAIL] Error: {e}")

print("\nNote: Both endpoints must be reachable from the SafeSpark sandbox.")
print("If Neo4j is reachable but the token endpoint is not, the JDBC OAuth")
print("factory will fail inside the sandbox even if Python can reach it.")

---

## Section 3: OAuth Token Acquisition Test

Validate that the OAuth credentials work by acquiring a token directly from the OIDC token endpoint. This test runs in Python (outside the SafeSpark sandbox) to isolate credential issues from JDBC driver issues.

**Expected Result**: PASS - Proves OAuth credentials are valid and the token endpoint is reachable.

In [ ]:
import requests
import json
import base64
import time
from datetime import datetime, timezone

print("=" * 60)
print("TEST: OAuth Token Acquisition (Client Credentials Grant)")
print("=" * 60)
print(f"\nToken Endpoint: {OAUTH_TOKEN_ENDPOINT}")
print(f"Client ID: {OAUTH_CLIENT_ID}")
print("Grant Type: client_credentials")

try:
    start_time = time.time()
    response = requests.post(
        OAUTH_TOKEN_ENDPOINT,
        data={
            "grant_type": "client_credentials",
            "client_id": OAUTH_CLIENT_ID,
            "client_secret": OAUTH_CLIENT_SECRET,
        },
        headers={"Content-Type": "application/x-www-form-urlencoded"},
    )
    elapsed = (time.time() - start_time) * 1000

    if response.status_code != 200:
        print(f"\n[FAIL] Token endpoint returned HTTP {response.status_code}")
        print(f"Response: {response.text}")
        print("\nStatus: FAIL")
    else:
        token_data = response.json()
        access_token = token_data["access_token"]

        # Decode JWT payload (no signature verification needed here)
        payload_b64 = access_token.split(".")[1]
        # Add padding
        payload_b64 += "=" * (4 - len(payload_b64) % 4)
        payload = json.loads(base64.urlsafe_b64decode(payload_b64))

        exp_time = datetime.fromtimestamp(payload.get("exp", 0), tz=timezone.utc)
        iss = payload.get("iss", "unknown")
        sub = payload.get("sub", "unknown")
        aud = payload.get("aud", "unknown")
        roles = payload.get("roles", [])

        print("\n" + "=" * 60)
        print(">>> TOKEN ACQUIRED SUCCESSFULLY <<<")
        print("=" * 60)
        print(f"\n[PASS] Token acquired in {elapsed:.1f}ms")
        print(f"\nJWT Claims:")
        print(f"  - Issuer (iss): {iss}")
        print(f"  - Subject (sub): {sub}")
        print(f"  - Audience (aud): {aud}")
        print(f"  - Roles: {roles}")
        print(f"  - Expires: {exp_time.isoformat()}")
        print(f"  - Token Type: {token_data.get('token_type', 'unknown')}")
        print(f"  - Expires In: {token_data.get('expires_in', 'unknown')}s")

        print("\n" + "-" * 60)
        print("RESULT: OAuth Client Credentials flow WORKING")
        print("        Token endpoint reachable, credentials valid")
        print("-" * 60)
        print("\nStatus: PASS")

except Exception as e:
    print(f"\n[FAIL] Token acquisition failed: {e}")
    print("\nStatus: FAIL")

---

## Section 4: Neo4j Bearer Token Connectivity Test

Validate that Neo4j accepts the bearer token by connecting with the Neo4j Python driver using the acquired token.

**Expected Result**: PASS - Proves Neo4j OIDC configuration is correct and tokens are accepted.

In [ ]:
import time
import requests
from neo4j import Auth, GraphDatabase

print("=" * 60)
print("TEST: Neo4j Python Driver with Bearer Token")
print("=" * 60)
print(f"\nTarget: {NEO4J_BOLT_URI}")
print("Auth: Bearer token from OAuth Client Credentials")

try:
    # Acquire fresh token
    token_response = requests.post(
        OAUTH_TOKEN_ENDPOINT,
        data={
            "grant_type": "client_credentials",
            "client_id": OAUTH_CLIENT_ID,
            "client_secret": OAUTH_CLIENT_SECRET,
        },
        headers={"Content-Type": "application/x-www-form-urlencoded"},
    )
    token_response.raise_for_status()
    access_token = token_response.json()["access_token"]

    start_time = time.time()
    driver = GraphDatabase.driver(
        NEO4J_BOLT_URI,
        auth=Auth("bearer", "", access_token),
    )

    driver.verify_connectivity()
    connect_time = (time.time() - start_time) * 1000

    print("\n" + "=" * 60)
    print(">>> BEARER TOKEN AUTHENTICATION SUCCESSFUL <<<")
    print("=" * 60)
    print(f"\n[PASS] Connected with bearer token in {connect_time:.1f}ms")

    with driver.session() as session:
        result = session.run("RETURN 1 AS test")
        record = result.single()
        print(f"[PASS] Query executed: RETURN 1 = {record['test']}")

        result = session.run("SHOW CURRENT USER YIELD user, roles RETURN user, roles")
        user_info = result.single()
        if user_info:
            print(f"\nAuthenticated User: {user_info['user']}")
            print(f"Assigned Roles: {user_info['roles']}")

    driver.close()

    print("\n" + "-" * 60)
    print("RESULT: Neo4j Bearer Token Authentication WORKING")
    print("        OIDC configuration correct, roles mapped")
    print("-" * 60)
    print("\nStatus: PASS")

except Exception as e:
    print(f"\n[FAIL] Bearer token connection failed: {e}")
    print("\nTroubleshooting:")
    print("  - Verify Neo4j OIDC configuration in neo4j.conf")
    print("  - Check that the token audience matches the OIDC provider audience")
    print("  - Verify role mappings (group_to_role_mapping)")
    print("\nStatus: FAIL")

---

## Section 5: Unity Catalog JDBC Connection (OAuth)

Create the UC JDBC connection with OAuth credentials. The key differences from a username/password connection:

1. The JDBC URL includes `authn.supplier=cc` and `authn.cc.tokenEndpoint=<url>`
2. The `user` field carries the OAuth client ID (not a Neo4j username)
3. The `password` field carries the OAuth client secret (not a Neo4j password)

The Neo4j JDBC driver reads `authn.supplier=cc` from the URL, loads the Client Credentials authentication factory via SPI, and uses the user/password as client credentials for the OAuth token exchange.

In [ ]:
# =============================================================================
# DIAGNOSTIC: Inspect CREATE CONNECTION SQL before execution
# =============================================================================
print("Connection name inspection:")
print(f"  Value: [{UC_CONNECTION_NAME}]")
print(f"  Length: {len(UC_CONNECTION_NAME)}")
print(f"  Repr: {repr(UC_CONNECTION_NAME)}")

print(f"\nJDBC URL inspection:")
print(f"  Length: {len(NEO4J_JDBC_URL_OAUTH)}")
print(f"  Contains single quotes: {chr(39) in NEO4J_JDBC_URL_OAUTH}")

print(f"\nFull CREATE SQL:")
print("-" * 60)
print(create_sql)
print("-" * 60)

print(f"\nComparison with notebook 01 pattern:")
print(f"  Has ENVIRONMENT block: {'ENVIRONMENT' in create_sql}")
print(f"  Has TYPE JDBC: {'TYPE JDBC' in create_sql}")
print(f"  Has OPTIONS: {'OPTIONS' in create_sql}")

In [ ]:
import time

print("=" * 60)
print("SETUP: Create Unity Catalog JDBC Connection (OAuth)")
print("=" * 60)
print(f"\nConnection Name: {UC_CONNECTION_NAME}")
print(f"JDBC URL: {NEO4J_JDBC_URL_OAUTH}")
print(f"Driver: org.neo4j.jdbc.Neo4jDriver")
print(f"Auth: OAuth Client Credentials (authn.supplier=cc)")

# Drop existing connection
spark.sql(f"DROP CONNECTION IF EXISTS {UC_CONNECTION_NAME}")
print(f"\n[INFO] Dropped existing connection (if any): {UC_CONNECTION_NAME}")

# Create connection with OAuth credentials
# user = OAuth client ID, password = OAuth client secret
# The JDBC driver reads authn.supplier=cc from the URL and uses the
# Client Credentials factory instead of basic auth
create_sql = f"""
CREATE CONNECTION {UC_CONNECTION_NAME} TYPE JDBC
ENVIRONMENT (
  java_dependencies '{JAVA_DEPENDENCIES}'
)
OPTIONS (
  url '{NEO4J_JDBC_URL_OAUTH}',
  user '{OAUTH_CLIENT_ID}',
  password '{OAUTH_CLIENT_SECRET}',
  driver 'org.neo4j.jdbc.Neo4jDriver',
  externalOptionsAllowList 'dbtable,query,partitionColumn,lowerBound,upperBound,numPartitions,fetchSize,customSchema'
)
"""

print(f"[INFO] Java Dependencies:")
print(f"  - {JDBC_JAR_PATH}")
print(f"[INFO] Auth Supplier: cc (Client Credentials)")
print(f"[INFO] Token Endpoint: {OAUTH_TOKEN_ENDPOINT}")

try:
    start_time = time.time()
    spark.sql(create_sql)
    create_time = (time.time() - start_time) * 1000

    print("\n" + "=" * 60)
    print(">>> UC CONNECTION CREATED (OAuth) <<<")
    print("=" * 60)
    print(f"\n[PASS] Connection '{UC_CONNECTION_NAME}' created in {create_time:.1f}ms")

    print(f"\nConnection Configuration:")
    print(f"  - Name: {UC_CONNECTION_NAME}")
    print(f"  - Type: JDBC")
    print(f"  - Driver: org.neo4j.jdbc.Neo4jDriver")
    print(f"  - Auth: OAuth Client Credentials (cc)")
    print(f"  - SQL Translation: Enabled")

except Exception as e:
    print(f"\n[FAIL] Failed to create connection: {e}")

In [ ]:
# Verify connection configuration
print("=" * 60)
print("VERIFY: Connection Configuration")
print("=" * 60)

try:
    df = spark.sql(f"DESCRIBE CONNECTION {UC_CONNECTION_NAME}")
    print("\nConnection details:")
    df.show(truncate=False)
except Exception as e:
    print(f"\n[FAIL] Cannot describe connection: {e}")

---

## Section 6: Unity Catalog JDBC Tests (OAuth)

These tests validate query patterns through the UC JDBC path with OAuth authentication. The queries are identical to notebook 01 -- only the authentication mechanism differs.

In [ ]:
# =============================================================================
# HELPER FUNCTION: Unity Catalog Query Test Runner
# =============================================================================
import time

def test_uc_query(test_name, sql_query, custom_schema, expected_cypher=None):
    """
    Execute and report on a Unity Catalog JDBC query test.

    Args:
        test_name: Name of the test
        sql_query: The SQL query to execute
        custom_schema: Schema string for customSchema option
        expected_cypher: Optional expected Cypher translation

    Returns:
        dict: {success: bool, df: DataFrame or None, time_ms: float}
    """
    print("=" * 60)
    print(f"TEST: UC OAuth - {test_name}")
    print("=" * 60)
    print(f"\nConnection: {UC_CONNECTION_NAME}")
    print(f"\nSQL Query:\n  {sql_query.strip()}")
    if expected_cypher:
        print(f"\nExpected Cypher:\n  {expected_cypher}")
    print(f"\nCustom Schema: {custom_schema}")

    try:
        start_time = time.time()
        df = spark.read.format("jdbc") \
            .option("databricks.connection", UC_CONNECTION_NAME) \
            .option("query", sql_query) \
            .option("customSchema", custom_schema) \
            .load()

        results = df.collect()
        total_time = (time.time() - start_time) * 1000

        print("\n" + "=" * 60)
        print(f">>> {test_name.upper()} WORKING <<<")
        print("=" * 60)
        print(f"\n[PASS] Query executed in {total_time:.1f}ms")

        print(f"\nQuery Results:")
        df.show(truncate=False)

        print(f"Execution Time: {total_time:.1f}ms")
        print("\nStatus: PASS")

        return {"success": True, "df": df, "time_ms": total_time, "results": results}

    except Exception as e:
        print(f"\n[FAIL] {test_name} failed:")
        print(f"\nError: {e}")
        print("\nStatus: FAIL")
        return {"success": False, "df": None, "time_ms": 0, "error": str(e)}

print("Helper function 'test_uc_query' defined successfully.")

In [ ]:
# Test: Basic Query
test_uc_query(
    test_name="Basic Query",
    sql_query="SELECT 1 AS test",
    custom_schema="test INT"
)

In [ ]:
# Test: remote_query() Function
import time

print("=" * 60)
print("TEST: UC OAuth - remote_query() Function")
print("=" * 60)
print(f"\nConnection: {UC_CONNECTION_NAME}")
print("Query: SELECT * FROM remote_query('connection', query => 'SELECT 1 AS test')")

try:
    start_time = time.time()
    df = spark.sql(f"""
        SELECT * FROM remote_query(
            '{UC_CONNECTION_NAME}',
            query => 'SELECT 1 AS test'
        )
    """)

    results = df.collect()
    total_time = (time.time() - start_time) * 1000
    value = results[0]['test'] if results else None

    print("\n" + "=" * 60)
    print(">>> UC remote_query() WITH OAUTH WORKING <<<")
    print("=" * 60)
    print(f"\n[PASS] remote_query() executed in {total_time:.1f}ms")

    print(f"\nQuery Results:")
    df.show()

    print(f"Result Value: {value}")
    print("\nStatus: PASS")

except Exception as e:
    print(f"\n[FAIL] remote_query() failed:")
    print(f"\nError: {e}")
    print("\nStatus: FAIL")

In [ ]:
# Test: COUNT Aggregate
test_uc_query(
    test_name="COUNT Aggregate",
    sql_query="SELECT COUNT(*) AS flight_count FROM Flight",
    custom_schema="flight_count LONG",
    expected_cypher="MATCH (n:Flight) RETURN count(n)"
)

In [ ]:
# Test: JOIN with Aggregate
test_uc_query(
    test_name="JOIN with Aggregate",
    sql_query="""SELECT COUNT(*) AS relationship_count
                 FROM Flight f
                 NATURAL JOIN DEPARTS_FROM r
                 NATURAL JOIN Airport a""",
    custom_schema="relationship_count LONG",
    expected_cypher="MATCH (f:Flight)-[:DEPARTS_FROM]->(a:Airport) RETURN count(*)"
)

In [ ]:
# Test: Aggregate with WHERE
test_uc_query(
    test_name="Aggregate with WHERE",
    sql_query="""SELECT COUNT(*) AS boeing_count
                 FROM Aircraft
                 WHERE manufacturer = 'Boeing'""",
    custom_schema="boeing_count LONG",
    expected_cypher="MATCH (n:Aircraft) WHERE n.manufacturer = 'Boeing' RETURN count(n)"
)

In [ ]:
# Test: Multiple Aggregates
test_uc_query(
    test_name="Multiple Aggregates (COUNT, MIN, MAX)",
    sql_query="""SELECT COUNT(*) AS total,
                        MIN(aircraft_id) AS first_id,
                        MAX(aircraft_id) AS last_id
                 FROM Aircraft""",
    custom_schema="total LONG, first_id STRING, last_id STRING"
)

In [ ]:
# Test: COUNT DISTINCT
test_uc_query(
    test_name="COUNT DISTINCT",
    sql_query="SELECT COUNT(DISTINCT manufacturer) AS unique_manufacturers FROM Aircraft",
    custom_schema="unique_manufacturers LONG"
)

In [ ]:
# Test: Authenticated Identity via UC JDBC
# Verifies that the OAuth SPI factory authenticates with the correct OIDC identity
# Note: SHOW CURRENT USER is a Cypher admin command, not SQL. It passes through
# the SQL translator (which returns null for unrecognized queries) and reaches
# Neo4j as native Cypher.
import time

print("=" * 60)
print("TEST: UC OAuth - Authenticated Identity")
print("=" * 60)
print(f"\nConnection: {UC_CONNECTION_NAME}")
print("Query: SHOW CURRENT USER YIELD user, roles RETURN user, roles")

try:
    start_time = time.time()
    df = spark.read.format("jdbc") \
        .option("databricks.connection", UC_CONNECTION_NAME) \
        .option("query", "SHOW CURRENT USER YIELD user, roles RETURN user, roles") \
        .option("customSchema", "user STRING, roles STRING") \
        .load()

    results = df.collect()
    total_time = (time.time() - start_time) * 1000

    print("\n" + "=" * 60)
    print(">>> AUTHENTICATED IDENTITY CONFIRMED <<<")
    print("=" * 60)
    print(f"\n[PASS] Identity query executed in {total_time:.1f}ms")

    print(f"\nAuthenticated Identity:")
    df.show(truncate=False)

    if results:
        print(f"  User: {results[0]['user']}")
        print(f"  Roles: {results[0]['roles']}")

    print("\nStatus: PASS")

except Exception as e:
    print(f"\n[FAIL] Identity query failed: {e}")
    print("\nNote: SHOW CURRENT USER requires Neo4j 4.4+ and passes through")
    print("as Cypher (not SQL). This may not work through the UC JDBC path")
    print("if Spark's query wrapping interferes with Cypher passthrough.")
    print("\nStatus: FAIL")

---

## Section 7: Fallback - Pre-acquired Bearer Token

If the SafeSpark sandbox cannot reach the OIDC token endpoint (network isolation), use this fallback: acquire the token in Python and pass it directly via `authScheme=BEARER` in the JDBC URL. This loses automatic token refresh but works for queries that complete within the token lifetime.

**Only use this section if Sections 5-6 fail with a network/connection error from the token endpoint.**

In [ ]:
# Fallback: Pre-acquire token and pass as bearer password
import time
import requests

print("=" * 60)
print("FALLBACK: Pre-acquired Bearer Token Connection")
print("=" * 60)
print("\nThis approach acquires the token in Python and passes it")
print("to the JDBC driver via authScheme=BEARER. No SPI factory needed.")

# Acquire token
token_response = requests.post(
    OAUTH_TOKEN_ENDPOINT,
    data={
        "grant_type": "client_credentials",
        "client_id": OAUTH_CLIENT_ID,
        "client_secret": OAUTH_CLIENT_SECRET,
    },
    headers={"Content-Type": "application/x-www-form-urlencoded"},
)
token_response.raise_for_status()
access_token = token_response.json()["access_token"]
expires_in = token_response.json().get("expires_in", "unknown")
print(f"\n[INFO] Token acquired (expires in {expires_in}s)")

# Build JDBC URL with BEARER auth scheme (no authn.supplier needed)
FALLBACK_JDBC_URL = (
    f"jdbc:neo4j://{NEO4J_HOST}:7687/{NEO4J_DATABASE}"
    f"?enableSQLTranslation=true"
    f"&authScheme=BEARER"
)

FALLBACK_CONNECTION = f"{UC_CONNECTION_NAME}_bearer"

spark.sql(f"DROP CONNECTION IF EXISTS {FALLBACK_CONNECTION}")

# For bearer auth, the password field carries the token
# The user field is ignored by the driver for bearer auth but UC requires a value
# Escape single quotes in the token to prevent SQL injection
escaped_token = access_token.replace("'", "''")

create_sql = f"""
CREATE CONNECTION {FALLBACK_CONNECTION} TYPE JDBC
ENVIRONMENT (
  java_dependencies '{JAVA_DEPENDENCIES}'
)
OPTIONS (
  url '{FALLBACK_JDBC_URL}',
  user 'bearer',
  password '{escaped_token}',
  driver 'org.neo4j.jdbc.Neo4jDriver',
  externalOptionsAllowList 'dbtable,query,partitionColumn,lowerBound,upperBound,numPartitions,fetchSize,customSchema'
)
"""

try:
    start_time = time.time()
    spark.sql(create_sql)
    create_time = (time.time() - start_time) * 1000
    print(f"[PASS] Fallback connection created in {create_time:.1f}ms")

    # Test query
    df = spark.read.format("jdbc") \
        .option("databricks.connection", FALLBACK_CONNECTION) \
        .option("query", "SELECT 1 AS test") \
        .option("customSchema", "test INT") \
        .load()

    results = df.collect()
    query_time = (time.time() - start_time) * 1000

    print("\n" + "=" * 60)
    print(">>> FALLBACK BEARER TOKEN CONNECTION WORKING <<<")
    print("=" * 60)
    print(f"\n[PASS] Query executed in {query_time:.1f}ms")
    df.show()

    print("\nNote: This connection uses a pre-acquired token.")
    print(f"The token expires in {expires_in}s. After expiry, recreate the connection.")
    print("\nStatus: PASS")

except Exception as e:
    print(f"\n[FAIL] Fallback connection failed: {e}")
    print("\nStatus: FAIL")